# Jak Wojtek uczy się chodzić

Wojtek to czworonożny robot, który zaczął jako **4BarBot** na Politechnice Wrocławskiej. Chodu nie ma zaprogramowanego: uczy się go metodą uczenia ze wzmocnieniem (RL) w symulacji, a wytrenowana sieć trafia potem na fizycznego robota. Ten notebook prowadzi przez tę drogę krok po kroku. Kroki z symulacją kończą się widokiem z MuJoCo.

| krok | co | czas na Colab T4 |
|---|---|---:|
| 0 | instalacja | 4 min |
| 1 | Wojtek stoi w MuJoCo | 1 min |
| 2 | polityka na początku treningu | 1 min |
| 3 | nagroda: co do niej wkładamy i jak steruje treningiem | czytanie |
| 4 | trening na GPU, próbka 10 mln kroków | 5 min |
| 5 | wytrenowana polityka w MuJoCo | 2 min |
| 6 | zadania: własne eksperymenty z nagrodą i komendami | 6 min każde |
| 7 | zadanie główne: Wojtek chodzi tam, gdzie każesz (panel sterowania) | 20 min |
| 8 | finał: Twoja polityka kontra ta z robota | 3 min |

Gdy brakuje czasu: zrób kroki 0–5, potem od razu krok 7, a zadania z kroku 6 na koniec.

## Robot

- 4 nogi, w każdej 3 silniki: odwodzenie biodra, biodro, kolano. Razem 12 silników sterowanych **pozycyjnie**: polityka podaje kąt docelowy, regulator PD w napędzie (na robocie MD80, w MuJoCo ten sam model serwa) zamienia go na moment.
- Nogi są czworobokami przegubowymi: poniżej silników łańcuch kinematyczny się zamyka. Za osobliwością kolana (ok. 3,2 rad) mechanizm może przeskoczyć do drugiej konfiguracji, dlatego cel kolana jest zawsze ograniczony.
- Masa 14 kg, wysokość stania ok. 0,125 m.
- Fizyka liczy się 250 razy na sekundę (krok 4 ms), polityka działa 50 razy na sekundę: jedna decyzja = 5 kroków fizyki.

## Skąd jest model

Wszystko leży w pakiecie ROS `ros/src/wojtek_description/`:

- `meshes/*.stl` — geometria ogniw robota z CAD-u;
- `urdf/*.urdf.xacro` — opis robota dla ROS-a (ten, którego używa prawdziwy robot i RViz);
- `mujoco/wojtek.xml` — ten sam robot zapisany w formacie MuJoCo (MJCF): ogniwa, przeguby, domknięcia czworoboków, siatki z `meshes/`, silniki i czujniki. Jest źródłem dla treningu;
- `mujoco/wojtek_mjx.xml` + `scene_mjx.xml` — wersja treningowa, **generowana** poleceniem `./training/run.sh build`. Skrypt bierze `wojtek.xml` i nanosi zmiany potrzebne w treningu: siatki przestają kolidować (stopy dostają kule, korpus prostopadłościan), korpus dostaje jawną masę, 12 silników momentowych staje się serwami PD, krok fizyki zostaje ustawiony na 4 ms. Tych plików nie edytuje się ręcznie; `scene_mjx.xml` dokłada podłogę, światło i kamerę śledzącą.

Notebook ładuje właśnie `scene_mjx.xml`, czyli dokładnie to, na czym trenuje polityka.

## Colab

- Środowisko: **GPU** (Runtime → Change runtime type → T4). Darmowy Colab nie zawsze przydziela GPU; bez niego działają kroki 0–3, a treningi trzeba odłożyć.
- Sesja wygasa po ok. 90 min bezczynności, a zerwana sesja kasuje wszystko na dysku maszyny (klon repozytorium, pakiety, treningi). Sam Restart session zachowuje pliki. Politykę z kroku 7 pobierz na dysk zaraz po eksporcie (komórka w kroku 7).
- Uruchamiaj komórki po kolei. Długie komórki (instalacja, trening, eksport) piszą, ile potrwają; nie przerywaj ich.

## Krok 0 — Instalacja

Klonuje repozytorium i instaluje `training/` (JAX, MuJoCo MJX, MJWarp, Brax). Trwa ok. 4 min. Jeśli następna komórka nie zaimportuje bibliotek, zrób Runtime → Restart session i uruchom od początku.

In [ ]:
import os, subprocess, sys
from pathlib import Path

if sys.platform == "linux":
    os.environ.setdefault("MUJOCO_GL", "egl")   # renderowanie bez ekranu; przed `import mujoco`

REPO_BRANCH = "Add-notebook-with-the-guidance-how-to-train-Wojtek"   # po scaleniu: "main"
REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "training" / "run.sh").exists()), None)
if REPO_ROOT is None:
    REPO_ROOT = Path.cwd() / "w01-tek"
    if not REPO_ROOT.exists():
        subprocess.run(["git", "clone", "-q", "-b", REPO_BRANCH, "https://github.com/machinekind/w01-tek.git", str(REPO_ROOT)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "-q", "--ff-only"])   # ponowne uruchomienie: dociągnij zmiany
TRAINING = REPO_ROOT / "training"

try:
    import wojtek_rl, fast_simplification, trimesh  # noqa: F401
except ImportError:
    import tomllib
    print("instalacja pakietów (ok. 4 min)...")
    lock = tomllib.loads((TRAINING / "uv.lock").read_text())
    mujoco_pin = next(p["version"] for p in lock["package"] if p["name"] == "mujoco")   # ta sama wersja co w locku
    res = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(TRAINING), f"mujoco=={mujoco_pin}",
                          "trimesh", "fast-simplification"],   # dwa ostatnie: uproszczone siatki do renderowania
                         capture_output=True, text=True)
    if res.returncode:
        print(res.stdout[-1500:], res.stderr[-3000:])
        raise SystemExit("pip install nie powiódł się (patrz wyżej)")
    # Colab ma preinstalowany nowszy plugin JAX dla CUDA 13; obok jax 0.9.2 tylko generuje błędy.
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "jax-cuda13-plugin", "jax-cuda13-pjrt"], capture_output=True)
    print("zainstalowano; jeśli następna komórka nie działa, zrestartuj sesję i uruchom od początku")
print(REPO_ROOT, "| python", sys.version.split()[0])

In [ ]:
import shutil
import mediapy as media
import mujoco
import numpy as np

if shutil.which("ffmpeg") is None:          # mediapy potrzebuje binarki ffmpeg
    import imageio_ffmpeg
    media.set_ffmpeg(imageio_ffmpeg.get_ffmpeg_exe())

for p in (TRAINING, REPO_ROOT / "learning"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
from wojtek_rl import paths
import wojtek_kurs as kurs

# Colab nie ma OpenGL od NVIDII: MuJoCo renderuje programowo, a pełne siatki robota
# (600 tys. trójkątów) kosztują 0,8 s na klatkę. Rysujemy więc kopię modelu z uproszczonymi
# siatkami (za pierwszym razem ok. 30 s); fizyka liczy się na oryginale.
widok = kurs.Widok(cache=TRAINING / "videos" / "guide" / "lowpoly")
frame = widok.klatka

def show(frames, fps=25):
    media.show_video(np.asarray(frames), fps=fps, codec="h264")

print("mujoco", mujoco.__version__, "| model:", paths.SCENE_XML.relative_to(REPO_ROOT),
      f"| siatki do rysowania: {int(widok.rmodel.mesh_facenum.sum()):,} trójkątów | GPU: {'jest' if kurs.jest_gpu() else 'BRAK'}")

## Krok 1 — Wojtek stoi w MuJoCo

Robot startuje z zapisanej pozy `home` i trzyma jej kąty w serwach PD ustawionych jak w treningu (kp 20, kd 1, limit 9 N·m). Ta poza jest odniesieniem dla polityki: akcja zero znaczy „stój tak”.

In [ ]:
model = kurs.model_z_serwem(kurs.SERWO)
data = mujoco.MjData(model)
mujoco.mj_resetDataKeyframe(model, data, model.key("home").id)
data.ctrl[:] = model.key("home").ctrl

frames = []
for k in range(int(4.0 / model.opt.timestep)):       # 4 s
    mujoco.mj_step(model, data)
    if k % 10 == 0:                                   # 25 klatek/s
        frames.append(frame(data))

print(f"serwa: {model.nu}, masa {sum(model.body_mass):.1f} kg, wysokość bazy {data.qpos[2]:.3f} m")
show(frames)

## Krok 2 — Polityka na początku treningu

Polityka (w PPO nazywana **aktorem**) to sieć neuronowa: na wejściu obserwacja, na wyjściu 12 przesunięć kątów względem pozy stania.

- **Obserwacja** (40 liczb): kąty i prędkości 12 przegubów, poprzednia akcja i **komenda** `[vx, vy, wz, wysokość]`, czyli zadana prędkość: do przodu, w bok, obrót, plus wysokość stania. Tylko to, co robot naprawdę mierzy: bez IMU, bez pozycji w świecie.
- **Akcja** (12 liczb w zakresie −1..1): cel = poza stania + akcja · skala, obcięty do limitów. Skala to 0,5 rad na przegub.
- **Sieć**: warstwy 512-256-128. W treningu do jej wyjścia dokłada się losowy szum, żeby próbowała różnych ruchów.

Tak wygląda start PPO: losowe wagi, szum, komenda 0,5 m/s do przodu. Sieć jeszcze nie wie, co komenda znaczy.

In [ ]:
KOMENDA = np.array([0.5, 0.0, 0.0, 0.125], np.float32)   # vx, vy, wz, wysokość stania
SEED = 0

U = kurs.ustawienia()                                      # serwo, skala akcji i limity z przepisu
model = kurs.model_z_serwem(U["serwo"])
data = mujoco.MjData(model)
mujoco.mj_resetDataKeyframe(model, data, model.key("home").id)
HOME = model.key("home").ctrl.copy()
qadr = np.array([model.jnt_qposadr[j] for j in model.actuator_trnid[:, 0]])
vadr = np.array([model.jnt_dofadr[j] for j in model.actuator_trnid[:, 0]])
SCALE, LOW, HIGH = U["skala"], U["dol"], U["gora"]
HIGH[2::3] = np.minimum(HIGH[2::3], 3.15)                  # bezpiecznik kolana (osobliwość 3,2 rad), jak w runtime robota

# Aktor jak w PPO na starcie: losowe wagi, wyjście = środek i rozrzut rozkładu akcji.
rng = np.random.default_rng(SEED)
sizes = [12 + 12 + 12 + 4, 512, 256, 128, 2 * 12]
W = [rng.uniform(-1, 1, (a, b)) * np.sqrt(3.0 / a) for a, b in zip(sizes[:-1], sizes[1:])]
def aktor(obs):
    x = obs
    for w in W[:-1]:
        x = x @ w; x = x / (1 + np.exp(-x))               # SiLU
    out = x @ W[-1]
    std = np.log1p(np.exp(out[12:])) + 1e-3               # softplus, jak w Brax
    return np.tanh(out[:12] + std * rng.standard_normal(12))

frames, last_act = [], np.zeros(12, np.float32)
x0 = data.qpos[0]
for i in range(200):                                       # 4 s przy 50 Hz
    obs = np.concatenate([data.qpos[qadr] - HOME, data.qvel[vadr], last_act, KOMENDA])
    last_act = aktor(obs).astype(np.float32)
    data.ctrl[:] = np.clip(HOME + last_act * SCALE, LOW, HIGH)
    for _ in range(5):
        mujoco.mj_step(model, data)
    if i % 2 == 0:
        frames.append(frame(data))
print(f"po 4 s: przebyte {data.qpos[0] - x0:+.2f} m w kierunku komendy (cel: +2.0 m), wysokość bazy {data.qpos[2]:.3f} m")
show(frames)

## Krok 3 — Nagroda

Trening nie mówi sieci, *jak* chodzić. Mówi tylko, ile punktów dostaje za każdy krok 20 ms, a PPO zmienia wagi tak, żeby suma punktów w epizodzie rosła. Nagroda to suma składników: `dt · Σ waga · składnik`. Co można do niej włożyć:

- **Zadanie**: `tracking_lin_vel`, `tracking_ang_vel` — jak blisko zadanej jest prędkość robota (kernel `exp(-błąd²/σ)`: 1 przy trafieniu, 0 daleko); `height_tracking` — trzymanie zadanej wysokości. To jedyne miejsce, gdzie komenda w ogóle ma znaczenie.
- **Postawa**: `orientation` (kara za przechył; aktor nie ma IMU, więc to jego jedyny „zmysł” pionu), `pose` (kara za odchylenie od pozy stania), `stand_still` i `stand_feet_down` (kary za ruch i uniesione stopy przy komendzie „stój”).
- **Chód**: `feet_air_time` (nagroda za czas stopy w powietrzu), `feet_phase` i `contact_match` (nagrody za zgodność stóp z wewnętrznym zegarem chodu, którego aktor nie widzi), `high_step` (unoszenie stóp), `feet_slip` (kara za poślizg stopy na ziemi).
- **Wysiłek i gładkość**: `torques`, `torque_rate`, `torque_limit` (kary za moment, jego skoki, dobijanie do limitu), `action_rate` (kara za skoki celów między krokami).
- **Koniec**: `termination` — kara za upadek, który kończy epizod.

Jak to wpływa na trening, w skrócie:

- Wagi to kompromis. Samo śledzenie komendy daje robota, który drży i grzeje silniki; sama gładkość daje robota, który stoi. Każdy składnik ma cenę w innych.
- Sieć znajdzie luki. Dużą karę za twarde lądowanie (składnik `feet_landing`, w tym przepisie wyłączony) omija, sunąc stopami zamiast stawiać kroki; kar naliczanych co krok potrafi „uniknąć”, przewracając się wcześnie, bo krótszy epizod to mniej kar. Dlatego obok nagrody patrzy się na długość epizodu.
- Czego nie ma w nagrodzie, tego nie będzie w chodzie: obroty w miejscu czy chód do tyłu pojawiają się dopiero, gdy komendy je zadają, a nagroda za nie płaci.

Poniżej wagi przepisu kursu `course_locomotion`: to przepis `locomotion` z tego repozytorium (od niego zaczynały pierwsze chodzące polityki Wojtka) z zegarem chodu usuniętym z obserwacji aktora, dzięki czemu polityka eksportuje się na robota, a w 50 mln kroków na T4 uczy się chodzić na komendę. Polityki przypięte na robocie trenowano innymi, nowszymi przepisami (inne serwo, inne wagi, miliardy kroków); w kroku 8 porównasz się z jedną z nich. Zero oznacza składnik wyłączony.

In [ ]:
cfg = kurs.srodowisko()          # przepis kursu: domyślne wartości środowiska + nakładka z YAML

GRUPY = {"zadanie": ["tracking_lin_vel", "tracking_ang_vel", "height_tracking"],
         "postawa": ["orientation", "pose", "stand_still", "stand_feet_down", "lin_vel_z", "ang_vel_xy"],
         "chód": ["feet_air_time", "feet_phase", "contact_match", "high_step", "feet_slip", "feet_apex", "feet_landing"],
         "wysiłek": ["torques", "torque_rate", "torque_limit", "action_rate", "action_accel", "energy"],
         "koniec": ["termination"]}
scales = cfg.reward.scales
for grupa, nazwy in GRUPY.items():
    print(f"{grupa:9s}", "  ".join(f"{n}={scales[n]:g}" for n in nazwy if n in scales))
c = cfg.command
print(f"komendy   vx {list(c.vx)} m/s  vy {list(c.vy)} m/s  wz {list(c.wz)} rad/s  wysokość {list(c.height)} m  | „stój” z p={c.zero_prob}")

## Krok 4 — Trening (GPU)

Ta sama sieć co w kroku 2, ale 4096 robotów naraz na GPU, każdy przez epizod 20 s, a po każdej porcji kroków PPO poprawia wagi w stronę większej nagrody. Co jakiś czas trener ocenia politykę i drukuje linię: `reward` to suma nagrody z epizodu, `ep_len` to jego długość w krokach (1000 = nie upadł). Dobry przebieg: `reward` rośnie, `ep_len` zostaje przy 1000. Pierwsza linia pojawia się po 1–3 min kompilacji. Colab może w tym czasie ostrzegać, że notebook „nie używa GPU”: używa go proces treningu, nie kernel; zignoruj.

Trening uruchamia `kurs.trenuj(nazwa, kroki, envs=4096, seed=0, dodatkowe=())`. Dokończony trening o tej samej nazwie nie startuje ponownie; przerwany jest kasowany i liczony od nowa. `dodatkowe` to nadpisania konfiguracji (Hydra), po jednym napisie:

| co | nadpisanie |
|---|---|
| waga składnika nagrody | `"++task.env.reward.scales.action_rate=-0.5"` |
| zakres komend | `"++task.env.command.vx=[0.3,0.6]"`, `"++task.env.command.wz=[0,0]"` |
| losowanie komendy „stój” | `"++task.env.command.zero_prob=0"` |
| pchnięcia w treningu | `"++task.env.push.vel=0.3"` |
| serwo | `"++task.env.pd_kp=60"`, `"++task.env.pd_kd=1.96"` (robot musiałby dostać to samo) |
| inny start | argument `seed=1` |

Budżet kroków a efekt na T4 (ok. 58 tys. kroków/s przy 4096 robotach, plus 1–3 min kompilacji), zmierzone tym przepisem:

| kroki | czas | co widać |
|---|---:|---|
| 10 mln | 3 min | robot stoi, zaczyna reagować na komendę |
| 50 mln | 15 min | chodzi na komendę: 1,8 m w 4 s przy 0,5 m/s, obroty w obie strony, krok w bok i do tyłu |
| 100–300 mln | 30–90 min | dokładniej trzyma prędkość, mniej drga |
| miliardy | godziny na H100 | polityki publikowane na robota |

Pierwszy trening to próbka: 10 mln kroków. Gdy trening kończy się błędem, `trenuj` wypisuje koniec `train.log`; jeśli jest tam mowa o `warp`, karta nie obsługuje MJWarp i trzeba dodać `dodatkowe=("+task.env.sim.backend=jax",)`.

In [ ]:
RUN = "proba_10m"
kurs.trenuj(RUN, kroki=10_000_000, envs=4096)

## Krok 5 — Wytrenowana polityka w MuJoCo

Eksport zamienia checkpoint na `policy.npz` (wagi) i `policy_meta.json` (kontrakt: układ obserwacji, skala akcji, limity, serwo) i sprawdza, że wersja NumPy daje to samo co sieć z treningu. Dokładnie te dwa pliki dostaje robot. `kurs.przebieg` to pętla z kroku 2 spakowana do funkcji: odczyt przegubów → `polityka.step` → cele do serw → 5 kroków fizyki. Ta sama komenda, wagi po treningu.

In [ ]:
from wojtek_rl.np_policy import load_policy_runtime

polityka = load_policy_runtime(kurs.eksportuj(RUN))
p = kurs.przebieg(polityka, komenda=(0.5, 0.0, 0.0), sekundy=4.0, widok=widok)
print(kurs.tabela({RUN: p}))
show(p["klatki"])

## Krok 6 — Zadania

Każde zadanie to jeden trening z inną konfiguracją i ten sam przebieg co wyżej. Nadaj każdemu własną nazwę `RUN`. 10 mln kroków starcza, żeby zobaczyć różnicę względem `proba_10m`; porównuj tabelą i wideo, nie samą nagrodą.

1. **Inny start.** `kurs.trenuj(RUN, kroki=10_000_000, seed=1)` z tą samą konfiguracją. Jak bardzo wynik zależy od losowości?
2. **Bez nagród za chód.** `feet_air_time=0`, `feet_phase=0`, `contact_match=0`. Czy robot w ogóle odrywa stopy, czy sunie?
3. **Gładkość.** `action_rate=-1.0` (4× większa kara). Co się dzieje z drganiami i z prędkością?
4. **Tylko do przodu.** `vx` w 0,3–0,6 m/s, `vy` i `wz` równe zeru i wyłączone losowanie „stój” (`zero_prob=0`). Sieć szybciej uczy się chodzić do przodu, ale tylko tego.

Szablon poniżej (ustawiony na zadanie 2); wpisz nazwę i nadpisania.

In [ ]:
RUN = "zadanie_2"
DODATKOWE = ("++task.env.reward.scales.feet_air_time=0", "++task.env.reward.scales.feet_phase=0",
             "++task.env.reward.scales.contact_match=0")
# zadanie 3: ("++task.env.reward.scales.action_rate=-1.0",)
# zadanie 4: ("++task.env.command.vx=[0.3,0.6]", "++task.env.command.vy=[0,0]", "++task.env.command.wz=[0,0]",
#             "++task.env.command.zero_prob=0")
kurs.trenuj(RUN, kroki=10_000_000, envs=4096, dodatkowe=DODATKOWE)

polityka = load_policy_runtime(kurs.eksportuj(RUN))
p = kurs.przebieg(polityka, komenda=(0.5, 0.0, 0.0), sekundy=4.0, widok=widok)
print(kurs.tabela({RUN: p}))
show(p["klatki"])

## Krok 7 — Zadanie główne: Wojtek chodzi tam, gdzie każesz

Wytrenuj politykę, która wykonuje dowolną komendę z zakresu przepisu: przód i tył, w bok, obrót, wysokość stania. To pełny zakres komend z kroku 3, więc potrzebuje więcej kroków niż próbka: 50 mln (ok. 15 min na T4); nie zamykaj karty. Panel niżej pozwala zadać komendę `vx, vy, wz` i wysokość i obejrzeć, jak polityka ją wykonuje; tabela mówi, jak daleko od komendy jest faktyczny ruch.

Warunek zaliczenia: przy `vx=0.5` robot przebywa ≥1,5 m w 4 s bez upadku, a przy `wz=0.8` i `wz=-0.8` wiersz „obrót łącznie” ma znak komendy i wartość ok. ±3 rad.

In [ ]:
RUN = "moj_wojtek"
kurs.trenuj(RUN, kroki=50_000_000, envs=4096)
moja = load_policy_runtime(kurs.eksportuj(RUN))
print("polityka:", moja.meta["run_name"], "| obserwacja:", moja.meta["obs_layout"])

In [ ]:
import ipywidgets as w
from IPython.display import display

vx = w.FloatSlider(0.5, min=-0.8, max=1.2, step=0.1, description="vx [m/s]")
vy = w.FloatSlider(0.0, min=-0.5, max=0.5, step=0.1, description="vy [m/s]")
wz = w.FloatSlider(0.0, min=-1.0, max=1.0, step=0.1, description="wz [rad/s]")
h = w.FloatSlider(0.125, min=0.09, max=0.17, step=0.01, description="wysokość [m]")
sek = w.IntSlider(4, min=2, max=10, description="sekundy")
przycisk, out = w.Button(description="Jedź"), w.Output()

def jedz(_):
    przycisk.disabled = True                       # przebieg i render trwają ok. 10 s
    with out:
        out.clear_output()
        p = kurs.przebieg(moja, komenda=(vx.value, vy.value, wz.value, h.value), sekundy=sek.value, widok=widok)
        print(kurs.tabela({moja.meta["run_name"]: p}))
        show(p["klatki"])
    przycisk.disabled = False

przycisk.on_click(jedz)
display(w.VBox([vx, vy, wz, h, sek, przycisk, out]))

Zapisz politykę poza Colabem: przycisk pakuje `policy.npz` i `policy_meta.json` i pobiera je na Twój komputer. W nowej sesji wgraj rozpakowany katalog przez panel plików Colaba i wczytaj go: `moja = load_policy_runtime("/content/moj_wojtek")`.

In [ ]:
from google.colab import files

pobierz = w.Button(description="Pobierz politykę (zip)")
pobierz.on_click(lambda _: files.download(shutil.make_archive(f"/content/{RUN}", "zip", kurs.eksportuj(RUN))))
display(pobierz)

## Krok 8 — Finał: Twoja polityka kontra ta z robota

Polityki, które przeszły do użytku, leżą w repozytoriach Hugging Face `<organizacja>/<nazwa>` (checkpoint, konfiguracja, testy, wideo i para `policy.npz` + `policy_meta.json`). Robot ładuje taką politykę po nazwie, tym samym kodem, którego używa tu `kurs.przebieg`. Repozytoria są prywatne: nazwę organizacji i token do odczytu dostaniesz od prowadzącego. W Colabie otwórz panel Secrets (ikona klucza po lewej), dodaj `HF_ORGANIZATION` i `HF_TOKEN` i włącz obu „Notebook access”. Bez tokena komórka porówna Twoją politykę z próbką `proba_10m`.

Domyślne odniesienie to polityka przypięta na robocie (`wojtek-quiet-locomotion`). Oba przebiegi dostają ten sam scenariusz: stój, idź 0,5 m/s, obróć się 0,7 rad/s, idź, stój. Porównuj: przebytą drogę, błąd prędkości i obrotu, drgania, upadki, moment. Polityka z robota ma za sobą 2 mld kroków, randomizację masy, tarcia i opóźnień oraz testy na sprzęcie; różnica pokazuje, ile z tego da się uzyskać w godzinę na T4.

In [ ]:
try:
    from google.colab import userdata
    for k in ("HF_ORGANIZATION", "HF_TOKEN"):
        try:
            os.environ.setdefault(k, userdata.get(k))
        except Exception:
            pass
except ImportError:
    pass
sys.path.insert(0, str(paths.WOJTEK_POLICY_PKG))
from wojtek_policy.policy_source import default_policy, resolve_policy

ORG = os.environ.get("HF_ORGANIZATION", "")
KEEPERS = ["wojtek-quiet-locomotion", "wojtek-stiff-locomotion", "wojtek-stiff-height-locomotion",
           "wojtek-stiff-locomotion-v2", "wojtek-stiff-kp80-locomotion", "wojtek-springy-locomotion-v2"]
if ORG:
    try:
        from huggingface_hub import HfApi
        KEEPERS = sorted(m.id.split("/", 1)[1] for m in HfApi().list_models(author=ORG)
                         if m.id.split("/", 1)[1].startswith("wojtek-")) or KEEPERS
    except Exception as e:
        print("lista z Hugging Face niedostępna:", type(e).__name__)
print("przypięta na robocie:", default_policy() or "(brak HF_ORGANIZATION w Secrets: porównanie z próbką proba_10m)")
wybor = w.Dropdown(options=KEEPERS, value="wojtek-quiet-locomotion" if "wojtek-quiet-locomotion" in KEEPERS else KEEPERS[0],
                   description="polityka")
display(wybor)

In [ ]:
if "moja" not in globals():                      # krok 7 pominięty lub sesja zerwana
    MOJA = "moj_wojtek" if kurs.status("moj_wojtek") == "complete" else "proba_10m"
    moja = load_policy_runtime(kurs.eksportuj(MOJA))
pin = default_policy()
ref = pin if pin and pin.split("/")[1].split("@")[0] == wybor.value else (f"{ORG}/{wybor.value}" if ORG else "")
if ref:
    odniesienie = load_policy_runtime(ref)
    nazwa = "robot"
    print("z robota:", resolve_policy(ref).source, "| serwo", odniesienie.meta["pd"])
else:
    odniesienie, nazwa = load_policy_runtime(kurs.eksportuj("proba_10m")), "proba_10m"

def scenariusz(i):                       # stój 2 s, przód 4 s, obrót 4 s, przód 4 s, stój 2 s
    if i < 100: return (0.0, 0.0, 0.0)
    if i < 300: return (0.5, 0.0, 0.0)
    if i < 500: return (0.0, 0.0, 0.7)
    if i < 700: return (0.5, 0.0, 0.0)
    return (0.0, 0.0, 0.0)

print("dwa przebiegi po 16 s z renderowaniem: ok. 1 min")
przebiegi = {"moja": kurs.przebieg(moja, scenariusz, sekundy=16, widok=widok),
             nazwa: kurs.przebieg(odniesienie, scenariusz, sekundy=16, widok=widok)}
print(kurs.tabela(przebiegi))
show(kurs.obok_siebie(przebiegi))        # po lewej Twoja, po prawej odniesienie

## Co dalej

- Więcej kroków, inne wagi, własne komendy: wszystko przez `kurs.trenuj(..., dodatkowe=...)`. Pełna lista nadpisań: `training/docs/configuration.md`.
- Ocena bez oglądania, z komórki notebooka: `!cd {TRAINING} && python -m wojtek_rl.report --run runs/moj_wojtek` (bateria testów) i `!cd {TRAINING} && python -m wojtek_rl.courses --run runs/moj_wojtek` (podążanie za ścieżką: proste, łuki, slalom, obroty). Wnioski z poprzednich iteracji: `skills/brax-locomotion-training/references/wojtek-training-lessons.md`.
- Na robota polityka trafia przez `./ros/deploy.sh --policy <organizacja/nazwa@commit>` i ręczne uzbrojenie. To krok człowieka, nigdy notebooka.